# Chapter 6: A Dynamic Array

The programs from Chapter 6 of *Data Structures and Algorithms in Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name. Benchmark timings differ on every computer and every run.

Run the cells from top to bottom (**Run > Run All Cells**). The programs import each other's files, so keep this notebook in its chapter folder.

## Fixed-Size Storage

`dynamic_array.py`

In [1]:
"""A dynamic array: a growable list built on fixed-size storage."""


def make_storage(capacity):
    """A fixed-size block of slots. We only ever read and write slots by
    index, never grow it, so it behaves like a raw block of memory."""
    return [None] * capacity


class DynamicArray:
    def __init__(self, growth=2.0):
        self._n = 0  # how many items are stored
        self._capacity = 1  # how many slots the storage has
        self._data = make_storage(1)
        self._growth = growth
        self.copies = 0  # items copied by resizing, for experiments

    def __len__(self):
        return self._n

    def _check(self, i):
        """Turn a negative index into a positive one; reject bad ones."""
        if i < 0:
            i += self._n
        if not 0 <= i < self._n:
            raise IndexError("DynamicArray index out of range")
        return i

    def __getitem__(self, i):
        return self._data[self._check(i)]

    def __setitem__(self, i, value):
        self._data[self._check(i)] = value

    def _resize(self, capacity):
        """Move every item into new storage with `capacity` slots."""
        new = make_storage(capacity)
        for i in range(self._n):
            new[i] = self._data[i]
        self.copies += self._n
        self._data, self._capacity = new, capacity

    def append(self, value):
        if self._n == self._capacity:  # full: grow first
            self._resize(
                max(self._n + 1, int(self._capacity * self._growth))
            )
        self._data[self._n] = value
        self._n += 1

    def insert(self, i, value):
        """Insert before position i, shifting later items right."""
        i = min(max(i + self._n if i < 0 else i, 0), self._n)
        if self._n == self._capacity:
            self._resize(
                max(self._n + 1, int(self._capacity * self._growth))
            )
        for j in range(self._n, i, -1):
            self._data[j] = self._data[j - 1]
        self._data[i] = value
        self._n += 1

    def pop(self, i=-1):
        """Remove and return the item at i, shifting later items left."""
        i = self._check(i)
        value = self._data[i]
        for j in range(i, self._n - 1):
            self._data[j] = self._data[j + 1]
        self._n -= 1
        self._data[self._n] = None  # don't keep a reference to it
        if 0 < self._n <= self._capacity // 4:  # mostly empty: shrink
            self._resize(self._capacity // 2)
        return value

    def __iter__(self):
        for i in range(self._n):
            yield self._data[i]

    def __repr__(self):
        return f"DynamicArray({list(self)})"

## Growing

`grow_demo.py`

In [2]:
"""Watch the capacity double as items are appended."""

from dynamic_array import DynamicArray

a = DynamicArray()
for i in range(17):
    before = a._capacity
    a.append(i)
    if a._capacity != before:
        print(
            f"append #{i + 1:>2}: capacity {before:>2} -> "
            f"{a._capacity:>2}, {a.copies:>2} items copied so far"
        )

append # 2: capacity  1 ->  2,  1 items copied so far
append # 3: capacity  2 ->  4,  3 items copied so far
append # 5: capacity  4 ->  8,  7 items copied so far
append # 9: capacity  8 -> 16, 15 items copied so far
append #17: capacity 16 -> 32, 31 items copied so far


## Why Doubling Works

`count_copies.py`

In [3]:
"""How many items does resizing copy in total? Three growth rules."""


def copies_for(n, grow):
    """Total items copied while appending n items, starting at 1 slot."""
    capacity, copies = 1, 0
    for size in range(n):
        if size == capacity:  # full: copy everything to new storage
            copies += size
            capacity = grow(capacity)
    return copies


rules = {
    "double": lambda c: 2 * c,
    "times 1.5": lambda c: max(c + 1, int(c * 1.5)),
    "add 100": lambda c: c + 100,
}
print(f"{'n':>9}" + "".join(f"{name:>16}" for name in rules))
for n in [1_000, 10_000, 100_000, 1_000_000]:
    row = "".join(f"{copies_for(n, g) / n:>16,.1f}" for g in rules.values())
    print(f"{n:>9,}{row}")
print("(copies per appended item)")

        n          double       times 1.5         add 100
    1,000             1.0             2.1             4.5
   10,000             1.6             2.4            49.5
  100,000             1.3             2.8           499.5
1,000,000             1.0             2.1         4,999.5
(copies per appended item)


## Timing It

`bench_append.py`

In [4]:
"""Appending n items: our DynamicArray against Python's list."""

import bench
from dynamic_array import DynamicArray


class AddHundred(DynamicArray):
    """Grows by a fixed 100 slots instead of doubling."""

    def append(self, value):
        if self._n == self._capacity:
            self._resize(self._capacity + 100)
        self._data[self._n] = value
        self._n += 1


def fill(make):
    def run(n):
        a = make()
        for i in range(n):
            a.append(i)

    return run


sizes = [5_000, 10_000, 20_000, 40_000, 80_000]
results = {
    "list": bench.doubling(fill(list), lambda n: (n,), sizes),
    "doubling": bench.doubling(fill(DynamicArray), lambda n: (n,), sizes),
    "add 100": bench.doubling(fill(AddHundred), lambda n: (n,), sizes),
}
bench.table(results)
bench.chart(results, "ch06_append.png", "Appending n items")

         n            list        doubling         add 100
     5,000           96 us          463 us         2.57 ms
    10,000     204 us x2.1     974 us x2.1    10.2 ms x4.0
    20,000     414 us x2.0    1.94 ms x2.0      40 ms x3.9
    40,000     826 us x2.0    4.01 ms x2.1     166 ms x4.1
    80,000    2.06 ms x2.5    8.15 ms x2.0     686 ms x4.1
